# Generate New Ground Truth from Corrected Annotations

This notebook builds a new YOLO training dataset using corrected annotation files from Label Studio.

It finalizes the annotation-training loop by creating accurate ground truth data for retraining.

🔧 Requirements

To run correctly, this notebook expects:

- A folder with the unannotated images:
  `partage/project_name/in/non_annotated_images/`

- A `labels.txt` file containing class definitions:
  `output/runs/train/model_folder/labels.txt`

- Corrected JSON annotation files:
  `partage/project_name/out/corrections/`

📌 The naming of `model_folder` and `result_folder` must follow this logic:

```text
project_name = projet01
model_folder = model01
→ result_folder = projet01_model01
```

Only project_name can be customized. Other folder names must follow expected naming patterns.

---

&copy; Marion Charpier - Use of this notebook requires appropriate citation.


## Environment

In [1]:
import os
import json
import shutil
import glob
import random

import pandas as pd

import sys
sys.path.append(os.path.join('..', 'modules'))

from transform_coordinates_functions import from_ls_to_yolo
from class_names_functions import get_labels, get_class_code
from folders_path import *
from manipulate_files import open_json_file, change_id

## Functions

### Create a new dataset with the correction files

In [2]:
def create_new_ground_truth(img_dataset_folder:str, yolo_model_folder:str, wanna_create:bool) -> None:
    """
    This function creates a new dataset based on inference corrections, allowing for the creation of an updated 
    ground truth dataset. The new dataset can be used to start a new training session. If the `wanna_create` 
    parameter is set to `False`, no new dataset is generated, and the function exits.
    
    :param img_dataset_folder: 
        - Type: str
        - Description: The absolute path to the folder containing the project images. This folder is used to 
                       locate the images that will be copied to the new dataset.
    :param yolo_model_folder: 
        - Type: str
        - Description: The path to the folder containing the YOLO model and its associated files. This folder 
                       is used to retrieve the `labels.txt` file and results folder for creating the new dataset.
    :param wanna_create: 
        - Type: bool
        - Description: A flag indicating whether to create the new dataset. If `True`, the new dataset is generated. 
                       If `False`, the function prints a message and exits without creating a dataset.
    
    :return: 
        - Type: None
        - Description: This function does not return a value. It creates a new folder containing the corrected 
                       labels and images for further training.

    This function is useful for managing and organizing datasets during iterative training and evaluation processes, 
    ensuring that corrected annotations are properly integrated into new training sessions.
    """

    if not wanna_create:
        print('No new dataset generated')
        return

    # Recompose the path to the model results folder
    results_folder = os.path.join(os.path.dirname(os.path.dirname(yolo_model_folder)), 
        'predict',
        img_dataset_folder.split('/')[-3] + '_' + os.path.basename(yolo_model_folder))
    
    # Create the folder for the new dataset
    new_folder = os.path.join(os.path.commonpath([img_dataset_folder, yolo_model_folder]), 'data', img_dataset_folder.split('/')[-3])

    # Check and create a unique folder
    counter = 1
    original_new_folder = new_folder
    
    while os.path.exists(new_folder):
        new_folder = f"{original_new_folder}_{counter}"
        counter += 1
    
    os.makedirs(new_folder)
    # Display the path of the new folder
    print("New Dataset Folder created:", new_folder)

    # Move corrected files to the new dataset folder
    corrections_folder = os.path.join(results_folder, 'correctedLabels')
    if os.path.exists(corrections_folder):
        shutil.move(corrections_folder, os.path.join(new_folder, 'labels'))
        print(f"Corrections folder found: {corrections_folder}")
    else:
        print(f"Corrections folder not found: {corrections_folder}")

    # Copy images to the new dataset
    os.makedirs(os.path.join(new_folder, 'images'))
    for file in os.listdir(img_dataset_folder):
        if file.endswith(('jpg','png','tiff')):
            shutil.copy(os.path.join(img_dataset_folder, file), os.path.join(new_folder, 'images', file))
    print(f"Images copied to {os.path.join(new_folder, 'images')}")

    # Copy the labels file
    shutil.copyfile(os.path.join(yolo_model_folder, 'labels.txt'), os.path.join(new_folder, 'labels.txt'))
    print(f"Labels file copied to: {os.path.join(new_folder, 'labels.txt')}")

    print(f'New dataset folder created')

### Move correction files and annotated images to the proper folders

In [3]:
def move_correction_files_and_images(img_dataset_folder:str) -> None:
    """
    This function moves images and correction JSON files to their respective folders based on a predefined directory structure. 
    It ensures that images are moved to an "annotated_images" folder, and corrected annotations are moved to an "annotations" folder, 
    with unique filenames for each annotation file.
    
    :param img_dataset_folder: 
        - Type: str
        - Description: The absolute path to the folder containing non-annotated images. This folder is used to locate 
                       the images and determine the destination paths for the annotated images and correction files.
    
    :return: 
        - Type: None
        - Description: This function does not return a value. It moves images and correction files to their designated 
                       folders and ensures that annotation filenames are unique.

    This function helps organize project data by structuring folders for annotated images and correction files, making 
    it easier to manage annotations and integrate them into the training workflow.
    """

    
    # Recompose the path of the "annotated_images" folder
    annotated_images_folder = img_dataset_folder.replace('non_annotated_images', 'annotated_images')
    
    # Recompose the path of the "corrections" folder
    corrections_folder = img_dataset_folder.replace('in/non_annotated_images', 'out/corrections')
    
    # Recompose the path of the "annotations" folder
    annotations_folder = img_dataset_folder.replace('in/non_annotated_images', 'out/annotations')

    # Move .jpg images to the annotated images folder
    for file in os.listdir(img_dataset_folder):
        if file.lower().endswith(('jpg', 'jpeg', 'png', 'tiff')):
            shutil.move(os.path.join(img_dataset_folder, file), os.path.join(annotated_images_folder, file))
    print(f"Images moved to {annotated_images_folder}")

    # Move correction files to the annotations folder
    for file in os.listdir(corrections_folder):
        if not file.startswith('.'):
            # Ensure unique file name
            new_annotation = os.path.join(annotations_folder, file)
            annotation_number = int(os.path.basename(file))
    
            while os.path.exists(new_annotation):
                annotation_number += 1
                new_annotation = os.path.join(annotations_folder, f"{annotation_number}")
                
            shutil.move(os.path.join(corrections_folder, file), new_annotation)

            # Changes the 'id' field in the JSON file to the basename of the file path
            change_id(new_annotation)
            
    print(f"Annotations files corrected and moved to {annotations_folder}")

### Add the image data to the pre-existing CSV (or create one)

In [4]:
def add_csv_data(img_dataset_folder:str, yolo_model_folder:str) -> None:
    """
    This function merges the CSV data of non-annotated images with the CSV file of annotated images. 
    It ensures that all relevant image metadata is consolidated into a single CSV file located in the 
    annotated images folder. If no CSV file exists in the annotated folder, the function moves the 
    CSV file from the non-annotated folder and updates the folder paths.
    
    :param img_dataset_folder: 
        - Type: str
        - Description: The absolute path to the folder containing the non-annotated images. 
                       This folder is used to locate the CSV file of non-annotated images.
    :param yolo_model_folder: 
        - Type: str
        - Description: The path to the folder containing the trained YOLO model. This folder is used to 
                       access the `labels.txt` file for retrieving class names and IDs.

    :return: 
        - Type: None
        - Description: This function does not return a value. It merges or moves the CSV file into the 
                       annotated images folder, ensuring data consistency.
    
    This function ensures that all image metadata is consolidated in a single CSV file, making it easier 
    to track and manage annotations and corrections across different stages of the project.
    """

    # Recompose the paths of the CSV files
    non_annotated_img_csv = os.path.join(img_dataset_folder, os.path.basename(img_dataset_folder.split('/')[-3]) + '.csv')

    annotated_images_folder = img_dataset_folder.replace('non_annotated_images', 'annotated_images')
    annotated_img_csv = os.path.join(annotated_images_folder, os.path.basename(annotated_images_folder.split('/')[-3]) + '.csv')

    # If no CSV is found, exit the function
    if not os.path.exists(non_annotated_img_csv) and not os.path.exists(annotated_img_csv):
        print("No CSV found in either folder")
        return

    # If only the annotated CSV exists, directly append the new data to it
    elif os.path.exists(annotated_img_csv) and not os.path.exists(non_annotated_img_csv):
        print("No non-annotated CSV file found. Adding images data directly")
        data = []

        images = [img for img in os.listdir(img_dataset_folder) if img.endswith(('jpg', 'png', 'tiff')) ]
        
        for file in images:
            img_name = file.split('.')[0]
            folder = img_dataset_folder
            with Image.open(os.path.join(img_dataset_folder, file)) as img:
                absolute_path = img.filename
                format = img.format
                width, height  = img.size
                img_size = width*height
        
            img_data = {
                  'Image_name' : img_name,
                  'Folder' : folder.replace('non_annotated', 'annotated'),
                  'Absolute_path' : absolute_path.replace('non_annotated', 'annotated'),
                  'Format' : format,
                  'Width' : width,
                  'Height': height,
                  'Image_size': int(width)*int(height)
            }
        
            data.append(img_data)
        
        df_annotated = pd.read_csv(annotated_img_csv, sep=';')
        
        #Créer un DataFrame à partir des nouvelles données collectées
        df_non_annotated = pd.DataFrame(data)
        
        # Concaténer les nouvelles données avec le DataFrame existant
        merged_df = pd.concat([df_annotated, df_non_annotated], ignore_index=True)
        
        # Sauvegarder le DataFrame fusionné dans le fichier annoté
        merged_df.to_csv(annotated_img_csv, sep=';', index=False)

    # If both CSVs exist, merge them
    else:
        # Open the files
        df_annotated = pd.read_csv(annotated_img_csv, sep=';')
        df_non_annotated = pd.read_csv(non_annotated_img_csv, sep=';')
        
        # Change the path from ‘non_annotated’ to ‘annotated’ to reflect the new image path.
        df_non_annotated['Folder'] = df_non_annotated['Folder'].str.replace('non_annotated', 'annotated')
        df_non_annotated['Absolute_path'] = df_non_annotated['Absolute_path'].str.replace('non_annotated', 'annotated')
        
        # Concatenate the two DataFrames by rows (add the data one after the other)
        merged_df = pd.concat([df_annotated, df_non_annotated], ignore_index=True)
        
        # Save the merged DataFrame in the annotated CSV file
        merged_df.to_csv(annotated_img_csv, sep=';', index=False)
        print(f'Merged data saved in {annotated_img_csv}')

        # Move the inference data CSV into the predict folder
        shutil.move(non_annotated_img_csv, os.path.join(get_results_folder(yolo_model_folder, img_dataset_folder), 'dataset.csv'))
        print(f"Inference data CSV moved to {os.path.join(get_results_folder(yolo_model_folder, img_dataset_folder), 'dataset.csv')}")

## Processing

### Enter absolute paths for variables

In [5]:
img_dataset_folder = 'ABSPATHTOTHEFOLDER' # To be changed. Absolute path to the folder named after your project.
yolo_model_folder = 'ABSPATHTOTHEMODELFOLDER' # To be changed. Asbolute path to the folder with the training data.

### Create the next dataset

In [ ]:
create_new_ground_truth(img_dataset_folder, yolo_model_folder, wanna_create=True)

### Move images and JSON files to the proper folders

In [ ]:
move_correction_files_and_images(img_dataset_folder)

### Add the image data to the pre-existing CSV (or create one)¶

In [ ]:
add_csv_data(img_dataset_folder, yolo_model_folder)